# 48h PM10+PM2.5（dyn）模型：与 `monthtail_2` 测试日对齐的按预报时效（lead）评估

**目的**
- 从 **`ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2`** 的 `meta_test.csv` 提取测试集日历日（与当前 S2 训练划分一致）。
- 在 **`PMST_s2_data_48h`（含 PM2.5+PM10 动态通道）** 产出的 48h 特征数据上推理，并按 `lead_hour` 统计模型指标随 lead 的变化。
- 读取 IFS 站点产品的 **0-48h 全部逐小时 lead**，与模型按同一条 lead 轴做对比，输出同图曲线。

**说明**
- 优先使用与 monthtail 同逻辑的 **`ml_dataset_fe_12h_48h_pm10_pm25_testonly_leadtime`**；否则回退到 **`ml_dataset_fe_12h_48h_pm10_pm25`**。
- **`dyn_vars_count=27`**，**Scaler**：`robust_scaler_w12_dyn27_s2_48h_pm10.pkl`。
- IFS 需要使用逐小时站点文件：**`IFS_VIS_0_48h_stations_2025_00_12.nc`**。
- **按起报分层**：`metrics_by_forecast_init.csv`、`fig7b_forecast_init_comparison.png`（需 `meta_test.csv` 含 **`init_time`**）；另 **`metrics_by_valid_hour.csv`**。
- **Lead curves (English labels)**：`metrics_by_lead_hour_init00Z.csv`, `metrics_by_lead_hour_init12Z.csv`, `fig_metrics_vs_lead_by_forecast_init_00Z_12Z.png`, `metrics_lead_mist_p_merge_vs_blend.csv`.


In [1]:
# ========= 路径与超参（按你的环境修改） =========
import os
import sys

try:
    _file = __file__
except NameError:
    _file = os.path.join(os.getcwd(), "paper_eval", "eval_48h_pm10_pm25_model_vs_ifs_standalone.ipynb")

BASE = os.path.dirname(os.path.dirname(os.path.abspath(_file)))
if not os.path.exists(os.path.join(BASE, "paper_eval")):
    BASE = os.path.dirname(BASE)

PAPER_EVAL_DIR = os.path.join(BASE, "paper_eval")
TRAIN_DIR = os.path.join(BASE, "train")
sys.path.insert(0, BASE)
sys.path.insert(0, TRAIN_DIR)
os.chdir(BASE)

# 12h monthtail：用于定义「测试日」集合
meta_test_12h_path = os.path.join(BASE, "ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2", "meta_test.csv")

# 48h 数据集：优先 testonly（与 monthtail 同划分）；否则为含 train/val/test 的目录并配合测试日过滤
data_48h_dir = os.path.join(BASE, "ml_dataset_fe_12h_48h_pm10_pm25_testonly_leadtime")
if not os.path.isdir(data_48h_dir):
    data_48h_dir = os.path.join(BASE, "ml_dataset_fe_12h_48h_pm10_pm25")

exp_id = "exp_1776227576_pm10_more_temp_search"
ckpt_path = os.path.join(BASE, "checkpoints", f"{exp_id}_S2_PhaseB_best_score.pt")
scaler_path = os.path.join(BASE, "checkpoints", "robust_scaler_w12_dyn27_s2_48h_pm10.pkl")
season_th_path = os.path.join(BASE, "checkpoints", f"{exp_id}_season_thresholds.pt")

output_dir = os.path.join(BASE, "paper_eval_results_48h_lead_monthtail_pm10_pm25")
os.makedirs(output_dir, exist_ok=True)

fog_th, mist_th = 0.10, 0.30  # 模型三分类阈值（概率）；IFS VIS_ifs 为米，脚本内单独用 500/1000m 与观测一致
no_calibration = True
threshold_rule = "mutual"
use_cpu = False
batch_size = 4096

# 仅评估某一类起报：None=全部；0=00Z；12=12Z（需 meta_test 含 init_time，见 PMST_s2_data_48h（pm10_pm25 特征）notebook）
FILTER_INIT_HOUR = None

window_size = 12
dyn_vars_count = 27
# 48h S2：extra = 31(fog_fe) + 4(cyclical) + 1(lead/48) = 36；meta 中另有 lead_hour、init_time
extra_feat_dim_expected = 36

ifs_all_leads_nc = os.path.join(BASE, "IFS_VIS_0_48h_stations_2025_00_12.nc")
run_ifs_baseline = os.path.isfile(ifs_all_leads_nc)

# Event-case maps (optional shapefile; None = outline omitted)
_shp_cand = os.path.join("/public/home/putianshu/中华人民共和国", "中华人民共和国.shp")
EVENT_SHP_PATH = _shp_cand if os.path.isfile(_shp_cand) else None
EVENT_FOOTPRINT_WINDOW_H = 6

print("BASE:", BASE)
print("12h meta_test:", meta_test_12h_path)
print("48h data_dir:", data_48h_dir)
print("ckpt:", ckpt_path)
print("scaler:", scaler_path)
print("output_dir:", output_dir)
print("IFS 0-48h baseline:", run_ifs_baseline, ifs_all_leads_nc if run_ifs_baseline else "")

BASE: /public/home/putianshu/vis_mlp
12h meta_test: /public/home/putianshu/vis_mlp/ml_dataset_s2_tianji_12h_pm10_pm25_monthtail_2/meta_test.csv
48h data_dir: /public/home/putianshu/vis_mlp/ml_dataset_fe_12h_48h_pm10_pm25
ckpt: /public/home/putianshu/vis_mlp/checkpoints/exp_1776227576_pm10_more_temp_search_S2_PhaseB_best_score.pt
scaler: /public/home/putianshu/vis_mlp/checkpoints/robust_scaler_w12_dyn27_s2_48h_pm10.pkl
output_dir: /public/home/putianshu/vis_mlp/paper_eval_results_48h_lead_monthtail_pm10_pm25
IFS 0-48h baseline: True /public/home/putianshu/vis_mlp/IFS_VIS_0_48h_stations_2025_00_12.nc


In [2]:
# ========= paper_eval.metrics_core + 动态导入 =========
import types
import importlib.util
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _name in list(sys.modules.keys()):
    if _name == "paper_eval" or _name.startswith("paper_eval."):
        del sys.modules[_name]

paper_eval_pkg = types.ModuleType("paper_eval")
paper_eval_pkg.__path__ = [PAPER_EVAL_DIR]
paper_eval_pkg.__file__ = os.path.join(PAPER_EVAL_DIR, "__init__.py")
sys.modules["paper_eval"] = paper_eval_pkg


def _load_sub(name):
    full = f"paper_eval.{name}"
    path = os.path.join(PAPER_EVAL_DIR, f"{name}.py")
    spec = importlib.util.spec_from_file_location(full, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full] = mod
    spec.loader.exec_module(mod)
    return mod


plot_style_mod = _load_sub("plot_style")
metrics_core_mod = _load_sub("metrics_core")
setup_paper_style = plot_style_mod.setup_paper_style
save_figure = plot_style_mod.save_figure

pred_from_thresholds = metrics_core_mod.pred_from_thresholds
pred_from_thresholds_mutual = getattr(metrics_core_mod, "pred_from_thresholds_mutual", pred_from_thresholds)
pred_from_joint_thresholds = getattr(metrics_core_mod, "pred_from_joint_thresholds", pred_from_thresholds)
compute_rare_event_report = metrics_core_mod.compute_rare_event_report

if hasattr(metrics_core_mod, "pred_from_season_thresholds"):
    pred_from_season_thresholds = metrics_core_mod.pred_from_season_thresholds
else:
    def pred_from_season_thresholds(
        probs, months, season_thresholds, default_fog_th=0.46, default_mist_th=0.38, rule="default"
    ):
        month_to_season = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
        months = np.asarray(months, dtype=np.int32).ravel()
        n = probs.shape[0]
        fog_th_arr = np.full(n, default_fog_th, dtype=np.float64)
        mist_th_arr = np.full(n, default_mist_th, dtype=np.float64)
        for i in range(n):
            season = month_to_season.get(int(months[i]))
            if season and season in season_thresholds:
                fog_th_arr[i] = season_thresholds[season]["fog_th"]
                mist_th_arr[i] = season_thresholds[season]["mist_th"]
        if rule == "mutual":
            return pred_from_thresholds_mutual(probs, fog_th_arr, mist_th_arr)
        if rule == "joint":
            return pred_from_joint_thresholds(probs, fog_th_arr, mist_th_arr)
        return pred_from_thresholds(probs, fog_th_arr, mist_th_arr)

from sklearn.metrics import average_precision_score

plot_scenarios_mod = _load_sub("plot_scenarios")
enrich_meta_forecast_init = getattr(plot_scenarios_mod, "enrich_meta_forecast_init", lambda m: m)
save_metrics_by_valid_hour = getattr(plot_scenarios_mod, "save_metrics_by_valid_hour", None)
plot_forecast_init_comparison = getattr(plot_scenarios_mod, "plot_forecast_init_comparison", None)
save_forecast_init_metrics_table = getattr(plot_scenarios_mod, "save_forecast_init_metrics_table", None)
derive_scenario_columns = getattr(plot_scenarios_mod, "derive_scenario_columns", None)
_compute_scenario_metrics = getattr(plot_scenarios_mod, "_compute_scenario_metrics", None)
plot_scenario_bars = getattr(plot_scenarios_mod, "plot_scenario_bars", None)
eval_scenario_detailed = getattr(plot_scenarios_mod, "eval_scenario_detailed", None)

plot_spatial_mod = _load_sub("plot_spatial")
detect_widespread_fog_events = plot_spatial_mod.detect_widespread_fog_events
load_china_shapefile = plot_spatial_mod.load_china_shapefile
plot_three_events_footprint_row = plot_spatial_mod.plot_three_events_footprint_row
plot_three_events_peak_row = plot_spatial_mod.plot_three_events_peak_row

# 本 notebook 的分层指标（init hour / valid hour）均直接使用上面的 preds，与 pred_from_rule 一致。
# 若将来调用 plot_scenario_bars / eval_scenario_detailed：请传入 meta["pred"]=preds，并设 threshold_rule=threshold_rule（与 pred_from_rule 相同），避免情景图与主评测混用不同阈值策略。

print("paper_eval metrics + plot_scenarios helpers loaded.")

paper_eval metrics + plot_scenarios helpers loaded.


In [3]:
# ========= 测试日、lead 解析、数据加载、推理（与 PMST_net_test_11_s2_pm10 / paper_eval 一致） =========


def load_test_calendar_dates_from_12h(meta_csv_path):
    """与训练用 monthtail 测试集一致的 UTC 日历日集合。"""
    m = pd.read_csv(meta_csv_path, parse_dates=["time"])
    dates = pd.to_datetime(m["time"]).dt.normalize().dt.date.unique()
    return set(dates), m


def infer_extra_feat_dim(n_cols, window_size, dyn_vars_count):
    """n_cols = win*dyn + 5(static_cont) + 1(veg) + extra"""
    base = window_size * dyn_vars_count + 5 + 1
    return int(n_cols - base)


def normalize_station_id(values):
    """统一 station_id 表示，避免 '12345' / '12345.0' / '0012345' 不匹配。"""
    s = pd.Series(values)
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^0+(\d+)$", r"\1", regex=True)
    return s.fillna("").values


def load_48h_xy_meta(data_dir, window_size, dyn_vars_count):
    Xp = os.path.join(data_dir, "X_test.npy")
    yp = os.path.join(data_dir, "y_test.npy")
    mp = os.path.join(data_dir, "meta_test.csv")
    if not all(os.path.exists(p) for p in (Xp, yp, mp)):
        raise FileNotFoundError(f"需要 X_test.npy / y_test.npy / meta_test.csv under {data_dir}")
    X = np.load(Xp, mmap_mode="r")
    y_raw = np.load(yp, mmap_mode="r")
    meta = pd.read_csv(mp, parse_dates=["time"])
    ex = infer_extra_feat_dim(X.shape[1], window_size, dyn_vars_count)
    return X, y_raw, meta, ex


def get_lead_hours(meta, X, window_size, dyn_vars_count, extra_feat_dim):
    """优先使用 meta_test.csv 中显式保存的 lead_hour；只有旧数据才回退到 X 推断。"""
    if "lead_hour" in meta.columns:
        lead = pd.to_numeric(meta["lead_hour"], errors="coerce").values.astype(np.float32)
        if np.isfinite(lead).any():
            return lead

    split_dyn = window_size * dyn_vars_count
    off = split_dyn + 6
    li = max(0, extra_feat_dim - 2)
    lead_norm = np.asarray(X[:, off + li], dtype=np.float64)
    return np.clip(lead_norm * 48.0, 0.0, 48.0).astype(np.float32)


def y_raw_to_cls(y_raw):
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if np.max(y_raw) < 100:
        y_raw = y_raw * 1000.0
    y_cls = np.zeros(len(y_raw), dtype=np.int64)
    y_cls[y_raw >= 500] = 1
    y_cls[y_raw >= 1000] = 2
    return y_cls, y_raw


def _regime_features_from_meta(meta_slice):
    month = np.asarray(meta_slice["month"].values, dtype=np.int32)
    hour = np.asarray(meta_slice["hour"].values, dtype=np.int32)
    lats = meta_slice["lat"].values if "lat" in meta_slice.columns else np.zeros(len(month))
    lons = meta_slice["lon"].values if "lon" in meta_slice.columns else np.zeros(len(month))
    is_coastal = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(np.float32)
    is_day = ((hour >= 6) & (hour < 18)).astype(np.float32)
    djf = ((month == 12) | (month <= 2)).astype(np.float32)
    mam = ((month >= 3) & (month <= 5)).astype(np.float32)
    jja = ((month >= 6) & (month <= 8)).astype(np.float32)
    son = ((month >= 9) & (month <= 11)).astype(np.float32)
    return np.column_stack([djf, mam, jja, son, is_day, is_coastal]).astype(np.float32)


def run_inference_48h(
    X,
    scaler,
    model,
    device,
    batch_size,
    window_size,
    dyn_vars_count,
    extra_feat_dim,
    temperature,
    meta,
):
    import torch
    import torch.nn.functional as F

    T = 1.0 if temperature is None or temperature == 1.0 else float(temperature)
    N = len(X)
    split_dyn = int(dyn_vars_count) * window_size
    # 与 PMST_net_test_11_s2_pm10._dyn_indices_log1p 一致：dyn>=27 时对末两维（PM2.5、PM10）做 log1p
    log_mask = np.zeros(split_dyn, dtype=bool)
    dyn_i = int(dyn_vars_count)
    log_idx = [2, 4, 9]
    if dyn_i >= 27:
        log_idx.extend([dyn_i - 2, dyn_i - 1])
    else:
        log_idx.append(dyn_i - 1)
    for t in range(window_size):
        base = t * dyn_i
        for i in log_idx:
            log_mask[base + i] = True

    use_cuda = device.type == "cuda"
    non_blocking = use_cuda
    need_regime = getattr(model, "regime_feat_dim", 0)
    if need_regime and meta is None:
        raise ValueError("model expects regime features")

    all_probs = []
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        rows = np.asarray(X[start:end], dtype=np.float32)
        feats = rows[:, : split_dyn + 5]
        feats[:, :split_dyn] = np.where(
            log_mask,
            np.log1p(np.maximum(feats[:, :split_dyn], 0)),
            feats[:, :split_dyn],
        )
        if scaler is not None:
            feats = (feats - scaler.center_) / (scaler.scale_ + 1e-6)
        veg = rows[:, split_dyn + 5 : split_dyn + 6]
        extra = rows[:, split_dyn + 6 :]
        if extra.shape[1] < extra_feat_dim:
            extra = np.pad(extra, ((0, 0), (0, extra_feat_dim - extra.shape[1])), mode="constant")
        elif extra.shape[1] > extra_feat_dim:
            extra = extra[:, :extra_feat_dim]
        final = np.concatenate([np.clip(feats, -10, 10), veg, np.clip(extra, -10, 10)], axis=1)
        if need_regime and meta is not None:
            regime = _regime_features_from_meta(meta.iloc[start:end])
            final = np.concatenate([final, regime], axis=1)
        final = np.nan_to_num(final, nan=0.0)
        x = torch.from_numpy(final).float().to(device, non_blocking=non_blocking)
        with torch.inference_mode():
            fine, _, _ = model(x)
            probs = F.softmax(fine / T, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def pred_from_rule(probs, months, season_thresholds, fog_th, mist_th, rule):
    if season_thresholds is not None:
        return pred_from_season_thresholds(probs, months, season_thresholds, fog_th, mist_th, rule=rule)
    if rule == "mutual":
        return pred_from_thresholds_mutual(probs, fog_th, mist_th)
    if rule == "joint":
        return pred_from_joint_thresholds(probs, fog_th, mist_th)
    return pred_from_thresholds(probs, fog_th, mist_th)


def pr_auc_ovr(probs, y_cls, c):
    y = (y_cls == c).astype(np.int32)
    if y.sum() == 0 or (1 - y).sum() == 0:
        return float("nan")
    return float(average_precision_score(y, probs[:, c]))


print("helpers OK.")

helpers OK.


In [4]:
# ========= 解析测试日、过滤 48h 样本、加载模型 =========
import torch

is_testonly_dataset = os.path.basename(data_48h_dir).endswith("testonly")

test_dates, meta_12h_ref = load_test_calendar_dates_from_12h(meta_test_12h_path)
pd.DataFrame(sorted(test_dates), columns=["test_calendar_date_utc"]).to_csv(
    os.path.join(output_dir, "monthtail_test_calendar_dates.csv"), index=False
)
print(f"[12h monthtail] unique test calendar days: {len(test_dates)}")

X, y_raw, meta, extra_dim = load_48h_xy_meta(data_48h_dir, window_size, dyn_vars_count)
print(f"[48h] X shape={X.shape}, inferred extra_feat_dim={extra_dim} (expected ~36)")
if extra_dim != extra_feat_dim_expected:
    print(f"[WARN] extra dim mismatch: got {extra_dim}, expected {extra_feat_dim_expected}")

meta = meta.copy()
meta["time"] = pd.to_datetime(meta["time"])
meta["month"] = meta["time"].dt.month
meta["hour"] = meta["time"].dt.hour
meta["date_utc"] = meta["time"].dt.date
meta["station_id_norm"] = normalize_station_id(meta["station_id"])
meta["sample_key"] = (
    meta["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
    + "__"
    + pd.Series(meta["station_id_norm"], dtype="string").fillna("")
)

meta_12h_ref = meta_12h_ref.copy()
meta_12h_ref["time"] = pd.to_datetime(meta_12h_ref["time"])
meta_12h_ref["station_id_norm"] = normalize_station_id(meta_12h_ref["station_id"])
meta_12h_ref["sample_key"] = (
    meta_12h_ref["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
    + "__"
    + pd.Series(meta_12h_ref["station_id_norm"], dtype="string").fillna("")
)
ref_keys = set(meta_12h_ref["sample_key"].tolist())

mask_day = meta["date_utc"].isin(test_dates)
mask_key = meta["sample_key"].isin(ref_keys)

if is_testonly_dataset:
    idx = np.where(mask_key.values)[0]
    print(f"[filter] testonly exact key overlap: {mask_key.mean():.2%} ({len(idx)} / {len(meta)})")
    if len(idx) == 0:
        idx = np.where(mask_day.values)[0]
        print(f"[WARN] exact key overlap is zero; fallback to test-day filter: {len(idx)} / {len(meta)}")
else:
    idx = np.where(mask_key.values)[0]
    print(f"[filter] exact monthtail sample overlap: {len(idx)} / {len(meta)}")
    if len(idx) == 0:
        idx = np.where(mask_day.values)[0]
        print(f"[WARN] exact key overlap is zero; fallback to test-day filter: {len(idx)} / {len(meta)}")

if len(idx) == 0:
    raise RuntimeError("无样本落在测试集内：请检查 48h 数据目录、station_id 归一化或 UTC 时间对齐。")

X_sub = X[idx]
y_raw_sub = np.asarray(y_raw[idx])
meta_sub = meta.iloc[idx].reset_index(drop=True)

if FILTER_INIT_HOUR is not None and "init_time" in meta_sub.columns:
    _ih = pd.to_datetime(
        meta_sub["init_time"].astype(str).str.replace(r"\.0$", "", regex=True),
        format="%Y%m%d%H",
        errors="coerce",
    ).dt.hour
    _keep = (_ih == int(FILTER_INIT_HOUR)).values
    n0 = len(meta_sub)
    X_sub = X_sub[_keep]
    y_raw_sub = y_raw_sub[_keep]
    meta_sub = meta_sub.iloc[np.where(_keep)[0]].reset_index(drop=True)
    print(f"[FILTER_INIT_HOUR={FILTER_INIT_HOUR}] kept {len(meta_sub)} / {n0} samples")
elif FILTER_INIT_HOUR is not None:
    print("[WARN] FILTER_INIT_HOUR set but meta_test has no init_time column; run updated dataset build.")

y_cls, y_raw_sub = y_raw_to_cls(y_raw_sub)

lead_h = get_lead_hours(meta_sub, X_sub, window_size, dyn_vars_count, extra_dim)
meta_sub["lead_hour"] = lead_h.astype(np.float32)
lead_hi = np.rint(lead_h).astype(np.int32)

meta_sub = enrich_meta_forecast_init(meta_sub)

n_dup = int(meta_sub["sample_key"].duplicated().sum())
print(f"[align] duplicate (time, station) keys in filtered 48h set: {n_dup}")
print(f"[lead] min/max (h): {lead_h.min():.2f} / {lead_h.max():.2f}; unique integer hours: {np.unique(lead_hi).size}")
print("[lead] first 20 unique hours:", sorted(np.unique(lead_hi))[:20])

align_diag = pd.DataFrame(
    {
        "metric": ["n_total_48h", "n_test_dates_48h", "n_exact_key_overlap", "n_filtered", "duplicate_time_station_keys"],
        "value": [len(meta), int(mask_day.sum()), int(mask_key.sum()), len(meta_sub), n_dup],
    }
)
align_diag.to_csv(os.path.join(output_dir, "alignment_diagnostics.csv"), index=False)

scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

if use_cpu or os.environ.get("CUDA_VISIBLE_DEVICES") == "":
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from PMST_net_test_11_s2_pm10 import ImprovedDualStreamPMSTNet

model = ImprovedDualStreamPMSTNet(
    dyn_vars_count=dyn_vars_count,
    window_size=window_size,
    hidden_dim=512,
    dropout=0.3,
    extra_feat_dim=extra_dim,
    num_classes=3,
).to(device)

state = torch.load(ckpt_path, map_location=device)
state = {k.replace("module.", ""): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()
if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

if "monthtail_2" in ckpt_path and "48h" not in os.path.basename(ckpt_path):
    print("[WARN] 当前 checkpoint 名称看起来不是专门的 48h 预报模型；若性能显著劣化，这更可能是输入分布不匹配，而非评估代码问题。")

season_thresholds = None
temperature = None
if (not no_calibration) and os.path.exists(season_th_path):
    try:
        try:
            cal = torch.load(season_th_path, map_location="cpu", weights_only=True)
        except TypeError:
            cal = torch.load(season_th_path, map_location="cpu")
        season_thresholds = cal.get("season_thresholds")
        temperature = cal.get("temperature")
        if temperature is not None:
            temperature = float(temperature)
        print("Calibration:", season_th_path, "T=", temperature)
    except Exception as e:
        print("[WARN] calibration load failed:", e)

probs = run_inference_48h(
    X_sub,
    scaler,
    model,
    device,
    batch_size,
    window_size,
    dyn_vars_count,
    extra_dim,
    temperature,
    meta_sub,
)

preds = pred_from_rule(
    probs,
    meta_sub["month"].values,
    season_thresholds,
    fog_th,
    mist_th,
    threshold_rule,
)

report_full = compute_rare_event_report(probs, y_cls, fog_th, mist_th, pred=preds)
print("--- Full subset (evaluation dataset, filtered) ---")
for k, v in report_full.items():
    if isinstance(v, (int, float)) and k in (
        "Fog_CSI", "Fog_R", "Fog_P", "Mist_CSI", "Mist_P", "Mist_R", "low_vis_precision", "false_positive_rate"
    ):
        print(f"  {k}: {v:.4f}")

[12h monthtail] unique test calendar days: 36


FileNotFoundError: 需要 X_test.npy / y_test.npy / meta_test.csv under /public/home/putianshu/vis_mlp/ml_dataset_fe_12h_48h_pm10_pm25

In [ ]:
# ========= 按整数 lead（起报后小时）聚合指标 =========
rows = []
for h in sorted(np.unique(lead_hi)):
    m = lead_hi == h
    if m.sum() < 50:
        continue
    ph, yh = probs[m], y_cls[m]
    pr = preds[m]
    rep = compute_rare_event_report(ph, yh, fog_th, mist_th, pred=pr)
    row = {
        "lead_hour": int(h),
        "n": int(m.sum()),
        "n_fog": int((yh == 0).sum()),
        "n_mist": int((yh == 1).sum()),
        "n_clear": int((yh == 2).sum()),
    }
    for key in (
        "Fog_CSI",
        "Fog_POD",
        "Fog_FAR",
        "Fog_R",
        "Fog_P",
        "Mist_CSI",
        "Mist_POD",
        "Mist_FAR",
        "Mist_R",
        "Mist_P",
        "low_vis_precision",
        "false_positive_rate",
        "macro_f1",
    ):
        if key in rep:
            row[key] = rep[key]
    row["PR_AUC_Fog"] = pr_auc_ovr(ph, yh, 0)
    row["PR_AUC_Mist"] = pr_auc_ovr(ph, yh, 1)
    rows.append(row)

lead_df = pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)
out_csv = os.path.join(output_dir, "metrics_by_lead_hour.csv")
lead_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
try:
    from IPython.display import display
except ImportError:
    display = print
display(lead_df)
print("[sanity] total by lead:", int(lead_df["n"].sum()), " / subset size:", len(meta_sub))

In [ ]:
# ========= 按起报时刻（init_time）与 valid hour 分层（与 run_paper_eval.py 对齐）=========
# 输出：metrics_by_forecast_init.csv、fig7b_forecast_init_comparison.png、metrics_by_valid_hour.csv

meta_en = enrich_meta_forecast_init(meta_sub.copy())

results_init = {}
if _compute_scenario_metrics is not None and "init_hour" in meta_en.columns and meta_en["init_hour"].notna().any():
    for h in sorted(meta_en["init_hour"].dropna().unique()):
        hi = int(h)
        m = (meta_en["init_hour"] == hi).values
        if m.sum() < 50:
            continue
        results_init["Forecast_init_{:02d}UTC".format(hi)] = _compute_scenario_metrics(y_cls[m], preds[m])
    if results_init and save_forecast_init_metrics_table is not None:
        p = os.path.join(output_dir, "metrics_by_forecast_init.csv")
        save_forecast_init_metrics_table(results_init, p)
        print("[Init] saved", p)
    if results_init and plot_forecast_init_comparison is not None:
        p = os.path.join(output_dir, "fig7b_forecast_init_comparison.png")
        _fig = plot_forecast_init_comparison(results_init, p)
        if _fig is not None:
            print("[Init] saved", p)
else:
    print("[Init] skip: meta_test has no init_time / init_hour (use per-init dataset build e.g. s2_data_per_init_split_pm10_pm25)")

if save_metrics_by_valid_hour is not None and derive_scenario_columns is not None:
    meta_v = derive_scenario_columns(meta_sub.copy())
    p = os.path.join(output_dir, "metrics_by_valid_hour.csv")
    save_metrics_by_valid_hour(y_cls, preds, meta_v, p)
    print("[Valid hour] saved", p)


In [ ]:
# ========= Forecast init (00Z / 12Z): P/R + pooled peak diagnostic figure =========
# Needs cell 5: lead_hi, probs, preds, y_cls, lead_df (optional); cell 3: pr_auc_ovr, compute_rare_event_report


def _lead_metrics_table_masked(lead_hi_arr, probs_arr, y_cls_arr, preds_arr, mask, min_n=50):
    rows = []
    for h in sorted(np.unique(lead_hi_arr[mask])):
        m = mask & (lead_hi_arr == h)
        if m.sum() < min_n:
            continue
        ph, yh = probs_arr[m], y_cls_arr[m]
        pr = preds_arr[m]
        rep = compute_rare_event_report(ph, yh, fog_th, mist_th, pred=pr)
        row = {
            "lead_hour": int(h),
            "n": int(m.sum()),
            "n_fog": int((yh == 0).sum()),
            "n_mist": int((yh == 1).sum()),
            "n_clear": int((yh == 2).sum()),
        }
        for key in (
            "Fog_CSI",
            "Fog_POD",
            "Fog_FAR",
            "Fog_R",
            "Fog_P",
            "Mist_CSI",
            "Mist_POD",
            "Mist_FAR",
            "Mist_R",
            "Mist_P",
            "low_vis_precision",
            "false_positive_rate",
            "macro_f1",
        ):
            if key in rep:
                row[key] = rep[key]
        row["PR_AUC_Fog"] = pr_auc_ovr(ph, yh, 0)
        row["PR_AUC_Mist"] = pr_auc_ovr(ph, yh, 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)


def _weighted_blend_by_lead(df0, df1, col):
    """Per-lead n-weighted blend across 00Z/12Z strata: P_blend = (P00*n00 + P12*n12)/(n00+n12)."""
    if df0.empty and df1.empty:
        return pd.DataFrame(columns=["lead_hour", col + "_blend"])
    a = (
        df0[["lead_hour", "n", col]].rename(columns={"n": "n_00", col: col + "_00"})
        if not df0.empty
        else pd.DataFrame(columns=["lead_hour", "n_00", col + "_00"])
    )
    b = (
        df1[["lead_hour", "n", col]].rename(columns={"n": "n_12", col: col + "_12"})
        if not df1.empty
        else pd.DataFrame(columns=["lead_hour", "n_12", col + "_12"])
    )
    m = a.merge(b, on="lead_hour", how="outer")
    m["n_00"] = m["n_00"].fillna(0)
    m["n_12"] = m["n_12"].fillna(0)
    vals = []
    for i in range(len(m)):
        s0, s1 = float(m["n_00"].iloc[i]), float(m["n_12"].iloc[i])
        if s0 + s1 <= 0:
            vals.append(np.nan)
            continue
        t = 0.0
        v0, v1 = m[col + "_00"].iloc[i], m[col + "_12"].iloc[i]
        if s0 > 0 and pd.notna(v0) and np.isfinite(v0):
            t += s0 * float(v0)
        if s1 > 0 and pd.notna(v1) and np.isfinite(v1):
            t += s1 * float(v1)
        vals.append(t / (s0 + s1))
    out = m[["lead_hour"]].copy()
    out[col + "_blend"] = vals
    return out


try:
    _meta_en = meta_en
except NameError:
    _meta_en = enrich_meta_forecast_init(meta_sub.copy())

b_m = pd.DataFrame()

try:
    lead_df_pooled = lead_df.copy()
except NameError:
    _all = np.ones(len(lead_hi), dtype=bool)
    lead_df_pooled = _lead_metrics_table_masked(lead_hi, probs, y_cls, preds, _all)

if "init_hour" not in _meta_en.columns or not _meta_en["init_hour"].notna().any():
    print("[Lead by forecast init] skip: meta has no init_hour (need init_time in meta_test.csv)")
    lead_df_init00 = pd.DataFrame()
    lead_df_init12 = pd.DataFrame()
else:
    mh = _meta_en["init_hour"].values
    m00 = np.isfinite(mh) & (mh == 0)
    m12 = np.isfinite(mh) & (mh == 12)
    lead_df_init00 = _lead_metrics_table_masked(lead_hi, probs, y_cls, preds, m00)
    lead_df_init12 = _lead_metrics_table_masked(lead_hi, probs, y_cls, preds, m12)
    p00 = os.path.join(output_dir, "metrics_by_lead_hour_init00Z.csv")
    p12 = os.path.join(output_dir, "metrics_by_lead_hour_init12Z.csv")
    lead_df_init00.to_csv(p00, index=False)
    lead_df_init12.to_csv(p12, index=False)
    print("[Lead by forecast init] saved:", p00, p12)
    print(f"  lead bins: 00Z={len(lead_df_init00)} rows, 12Z={len(lead_df_init12)} rows")

    C_MERGED = "#1f2937"
    C_00 = "#0f766e"
    C_12 = "#c2410c"
    LW_M, LW_S = 2.4, 1.8

    setup_paper_style(font_family="DejaVu Sans")
    fig = plt.figure(figsize=(13.5, 11.2))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.15, 1.15, 1.0], hspace=0.35, wspace=0.22)

    def _plot_pr_family(ax, col_p, col_r, title):
        if not lead_df_pooled.empty:
            ax.plot(
                lead_df_pooled["lead_hour"],
                lead_df_pooled[col_p],
                color=C_MERGED,
                lw=LW_M,
                ls="-",
                label=f"Pooled {col_p}",
                zorder=3,
            )
            ax.plot(
                lead_df_pooled["lead_hour"],
                lead_df_pooled[col_r],
                color=C_MERGED,
                lw=LW_M,
                ls=":",
                label=f"Pooled {col_r}",
                zorder=3,
            )
        if not lead_df_init00.empty:
            ax.plot(lead_df_init00["lead_hour"], lead_df_init00[col_p], "o-", color=C_00, lw=LW_S, ms=4, label=f"00Z {col_p}")
            ax.plot(
                lead_df_init00["lead_hour"],
                lead_df_init00[col_r],
                "o--",
                color=C_00,
                lw=LW_S,
                ms=3,
                alpha=0.88,
                label=f"00Z {col_r}",
            )
        if not lead_df_init12.empty:
            ax.plot(lead_df_init12["lead_hour"], lead_df_init12[col_p], "s-", color=C_12, lw=LW_S, ms=4, label=f"12Z {col_p}")
            ax.plot(
                lead_df_init12["lead_hour"],
                lead_df_init12[col_r],
                "s--",
                color=C_12,
                lw=LW_S,
                ms=3,
                alpha=0.88,
                label=f"12Z {col_r}",
            )
        ax.set_ylabel("Score")
        ax.set_title(title, fontsize=11, fontweight="600", pad=6)
        ax.grid(True, alpha=0.28, ls="-", lw=0.6)
        ax.legend(fontsize=7, ncol=2, loc="best", framealpha=0.92)

    ax0 = fig.add_subplot(gs[0, 0])
    _plot_pr_family(ax0, "Fog_P", "Fog_R", "Fog — precision / recall vs lead")
    ax0.set_xlabel("Lead time (h)")

    ax1 = fig.add_subplot(gs[0, 1])
    _plot_pr_family(ax1, "Mist_P", "Mist_R", "Mist — precision / recall vs lead")
    ax1.set_xlabel("Lead time (h)")

    ax2 = fig.add_subplot(gs[1, 0])
    if not lead_df_pooled.empty:
        ax2.plot(lead_df_pooled["lead_hour"], lead_df_pooled["Fog_CSI"], color=C_MERGED, lw=LW_M, label="Pooled Fog CSI")
        ax2.plot(lead_df_pooled["lead_hour"], lead_df_pooled["Mist_CSI"], color=C_MERGED, lw=LW_M, ls=":", label="Pooled Mist CSI")
    if not lead_df_init00.empty:
        ax2.plot(lead_df_init00["lead_hour"], lead_df_init00["Fog_CSI"], "o-", color=C_00, lw=LW_S, ms=3, label="00Z Fog CSI")
        ax2.plot(lead_df_init00["lead_hour"], lead_df_init00["Mist_CSI"], "o--", color=C_00, lw=LW_S, ms=2, label="00Z Mist CSI")
    if not lead_df_init12.empty:
        ax2.plot(lead_df_init12["lead_hour"], lead_df_init12["Fog_CSI"], "s-", color=C_12, lw=LW_S, ms=3, label="12Z Fog CSI")
        ax2.plot(lead_df_init12["lead_hour"], lead_df_init12["Mist_CSI"], "s--", color=C_12, lw=LW_S, ms=2, label="12Z Mist CSI")
    ax2.set_ylabel("CSI")
    ax2.set_xlabel("Lead time (h)")
    ax2.set_title("CSI — pooled vs split by init", fontsize=11, fontweight="600")
    ax2.grid(True, alpha=0.28)
    ax2.legend(fontsize=7, ncol=2, loc="best", framealpha=0.92)

    ax3 = fig.add_subplot(gs[1, 1])
    if not lead_df_pooled.empty:
        ax3.plot(lead_df_pooled["lead_hour"], lead_df_pooled["low_vis_precision"], color=C_MERGED, lw=LW_M, label="Pooled low-vis P")
        ax3.plot(
            lead_df_pooled["lead_hour"],
            lead_df_pooled["false_positive_rate"],
            color=C_MERGED,
            lw=LW_M,
            ls=":",
            label="Pooled Clear FPR",
        )
    if not lead_df_init00.empty:
        ax3.plot(lead_df_init00["lead_hour"], lead_df_init00["low_vis_precision"], "o-", color=C_00, lw=LW_S, ms=3, label="00Z low-vis P")
    if not lead_df_init12.empty:
        ax3.plot(lead_df_init12["lead_hour"], lead_df_init12["low_vis_precision"], "s-", color=C_12, lw=LW_S, ms=3, label="12Z low-vis P")
    ax3.set_ylabel("Rate")
    ax3.set_xlabel("Lead time (h)")
    ax3.set_title("Low-vis precision / clear FPR", fontsize=11, fontweight="600")
    ax3.grid(True, alpha=0.28)
    ax3.legend(fontsize=7, loc="best", framealpha=0.92)

    ax4 = fig.add_subplot(gs[2, 0])
    leads_u = sorted(set(lead_df_init00["lead_hour"].tolist()) | set(lead_df_init12["lead_hour"].tolist()))
    n00 = [
        int(lead_df_init00.loc[lead_df_init00["lead_hour"] == h, "n"].sum()) if h in lead_df_init00["lead_hour"].values else 0
        for h in leads_u
    ]
    n12 = [
        int(lead_df_init12.loc[lead_df_init12["lead_hour"] == h, "n"].sum()) if h in lead_df_init12["lead_hour"].values else 0
        for h in leads_u
    ]
    x = np.arange(len(leads_u))
    w = 0.42
    ax4.bar(x - w / 2, n00, width=w, color=C_00, label="n (00Z)", alpha=0.88, edgecolor="white", lw=0.4)
    ax4.bar(x + w / 2, n12, width=w, color=C_12, label="n (12Z)", alpha=0.88, edgecolor="white", lw=0.4)
    step = max(1, len(leads_u) // 16)
    ax4.set_xticks(x[::step])
    ax4.set_xticklabels([str(leads_u[i]) for i in range(0, len(leads_u), step)], fontsize=7)
    ax4.set_ylabel("Count")
    ax4.set_xlabel("Lead time (h)")
    ax4.set_title("Samples per lead (00Z vs 12Z; drives pooled weighting)", fontsize=10, fontweight="600")
    ax4.legend(fontsize=8, loc="upper right")
    ax4.grid(True, axis="y", alpha=0.25)

    ax5 = fig.add_subplot(gs[2, 1])
    b_m = _weighted_blend_by_lead(lead_df_init00, lead_df_init12, "Mist_P")
    if not lead_df_pooled.empty and not b_m.empty and "Mist_P_blend" in b_m.columns:
        ax5.plot(lead_df_pooled["lead_hour"], lead_df_pooled["Mist_P"], color=C_MERGED, lw=LW_M, label="Pooled Mist_P (full sample)")
        merged = lead_df_pooled[["lead_hour", "Mist_P"]].merge(b_m, on="lead_hour", how="inner")
        ax5.plot(merged["lead_hour"], merged["Mist_P_blend"], color="#7c3aed", lw=2.0, ls="--", label="n-weighted blend (00Z+12Z)")
        diff = merged["Mist_P"].values - merged["Mist_P_blend"].values
        ax5.fill_between(merged["lead_hour"], 0, diff, color="#94a3b8", alpha=0.35, label="Pooled − blend (diff)")
    ax5.set_xlabel("Lead time (h)")
    ax5.set_ylabel("Mist precision")
    ax5.set_title("Mist P: pooled vs n-weighted blend (close curves => mixture-like)", fontsize=10, fontweight="600")
    ax5.grid(True, alpha=0.28)
    ax5.legend(fontsize=7, loc="best", framealpha=0.92)

    fig.suptitle(
                "48h PM10+PM2.5 (dyn) — precision / recall & pooled-curve diagnostics (00Z vs 12Z)\n"
        "Solid ≈ precision, dotted ≈ recall; dark gray = pooled full sample.",
        fontsize=12,
        fontweight="600",
        y=0.995,
    )
    fig.text(
        0.5,
        0.014,
        "Pooled curve = one metric on all samples. If it matches the n-weighted blend of 00Z/12Z, peaks are mostly from"
        " weighting + shape mix across leads; a large gap means more than a simple mixture (class mix differs).",
        ha="center",
        fontsize=8.5,
        color="#4b5563",
    )
    fig.subplots_adjust(top=0.90, left=0.07, right=0.98, bottom=0.11)

    fig_path = os.path.join(output_dir, "fig_metrics_vs_lead_by_forecast_init_00Z_12Z.png")
    save_figure(fig, fig_path)
    plt.show()
    print("[Lead by forecast init] figure:", fig_path)

    try:
        if b_m is not None and not b_m.empty and not lead_df_pooled.empty and "Mist_P_blend" in b_m.columns:
            chk = lead_df_pooled[["lead_hour", "Mist_P", "n"]].merge(b_m, on="lead_hour", how="inner")
            chk["diff_merge_minus_blend"] = chk["Mist_P"] - chk["Mist_P_blend"]
            chk_path = os.path.join(output_dir, "metrics_lead_mist_p_merge_vs_blend.csv")
            chk.to_csv(chk_path, index=False)
            print("[Lead by forecast init] diagnostic CSV:", chk_path)
    except Exception as _e:
        print("[Lead by forecast init] skip blend CSV:", _e)


In [ ]:
# ========= 曲线图（IFS 48h 对比点会在下一格生成后叠加） =========
setup_paper_style()
ifs_point = None
if lead_df.empty:
    print("lead_df is empty: not enough samples or per-lead bins below min count.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    x = lead_df["lead_hour"].values

    axes[0, 0].plot(x, lead_df["Fog_CSI"], "o-", label="Model Fog CSI")
    axes[0, 0].plot(x, lead_df["Mist_CSI"], "s-", label="Model Mist CSI")
    axes[0, 0].set_ylabel("CSI")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(x, lead_df["Fog_R"], "o-", label="Model Fog Recall")
    axes[0, 1].plot(x, lead_df["Fog_P"], "^-", label="Model Fog Precision")
    axes[0, 1].plot(x, lead_df["Mist_R"], "s-", label="Model Mist Recall")
    axes[0, 1].plot(x, lead_df["Mist_P"], "d-", label="Model Mist Precision")
    axes[0, 1].set_ylabel("Precision / Recall")
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(x, lead_df["PR_AUC_Fog"], "o-", label="Model PR-AUC Fog")
    axes[1, 0].plot(x, lead_df["PR_AUC_Mist"], "s-", label="Model PR-AUC Mist")
    axes[1, 0].set_ylabel("PR-AUC")
    axes[1, 0].set_xlabel("Lead time (h)")
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(x, lead_df["low_vis_precision"], "o-", label="Model Low-vis precision")
    axes[1, 1].plot(x, lead_df["false_positive_rate"], "s-", label="Model FPR (Clear)")
    axes[1, 1].set_ylabel("Rate")
    axes[1, 1].set_xlabel("Lead time (h)")
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)

    fig.suptitle("48h PM10+PM2.5 (dyn) model — metrics vs lead")
    fig.tight_layout()
    fig_path = os.path.join(output_dir, "fig_metrics_vs_lead_hour_model_only.png")
    save_figure(fig, fig_path)
    plt.show()

In [ ]:
# ========= 模型 vs IFS：独立完整对比（内联实现，无需其他前置导入） =========
import os
from typing import Callable, Dict, Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


def normalize_station_id(values: Iterable) -> pd.Series:
    s = pd.Series(values, dtype="string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^0+(?=\d)", "", regex=True)
    return s.fillna("")


def vis_meters_to_obs_cls(vis_m: float) -> int:
    """IFS `VIS_ifs` 为米（见 get_IFS_vis_f48_stations.py attrs）；与 y_cls 分界一致。"""
    if not np.isfinite(vis_m):
        return -1
    if vis_m < 500.0:
        return 0
    if vis_m < 1000.0:
        return 1
    return 2


def build_model_side(meta_sub: pd.DataFrame, y_cls: np.ndarray) -> pd.DataFrame:
    df = meta_sub.copy()
    if "time" not in df.columns or "lead_hour" not in df.columns:
        raise ValueError("meta_sub must contain 'time' and 'lead_hour'")

    if "station_id_norm" not in df.columns:
        if "station_id" not in df.columns:
            raise ValueError("meta_sub must contain 'station_id_norm' or 'station_id'")
        df["station_id_norm"] = normalize_station_id(df["station_id"])
    else:
        df["station_id_norm"] = normalize_station_id(df["station_id_norm"])

    df["time"] = pd.to_datetime(df["time"])
    df["lead_hour"] = pd.to_numeric(df["lead_hour"], errors="coerce")
    df = df[df["lead_hour"].notna()].copy()
    df["lead_hour"] = np.rint(df["lead_hour"]).astype(np.int32)

    y_cls = np.asarray(y_cls).reshape(-1)
    if len(y_cls) != len(df):
        raise ValueError(
            f"len(y_cls)={len(y_cls)} != len(meta_sub after lead filter)={len(df)}"
        )

    df["obs_cls"] = y_cls.astype(np.int32)
    df = df[df["obs_cls"].isin([0, 1, 2])].copy()

    df["join_key"] = (
        df["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
        + "__"
        + df["station_id_norm"]
        + "__"
        + df["lead_hour"].astype(str)
    )
    return df.reset_index(drop=True)


def recover_valid_time_grid(
    ds_ifs: xr.Dataset, n_rec: int, n_lead: int, lead_vals: np.ndarray
) -> np.ndarray:
    lead_vals = np.asarray(lead_vals).reshape(-1).astype(np.int32)
    if lead_vals.shape[0] != n_lead:
        raise ValueError(f"lead_vals len={lead_vals.shape[0]} != n_lead={n_lead}")

    if "valid_time" in ds_ifs.coords:
        vt_raw = ds_ifs["valid_time"].values
        vt_arr = np.asarray(vt_raw)

        if vt_arr.ndim == 2:
            if vt_arr.shape == (n_rec, n_lead):
                return pd.to_datetime(vt_arr).values.astype("datetime64[ns]")
            if vt_arr.shape == (n_lead, n_rec):
                return pd.to_datetime(vt_arr.T).values.astype("datetime64[ns]")
            raise ValueError(f"2D valid_time shape={vt_arr.shape}, expected {(n_rec, n_lead)}")

        if vt_arr.ndim == 1 and vt_arr.shape[0] == n_rec * n_lead:
            return pd.to_datetime(vt_arr).values.astype("datetime64[ns]").reshape(n_rec, n_lead)

        vt_1d = vt_arr.reshape(-1)
        if vt_1d.shape[0] == n_rec:
            base = pd.to_datetime(vt_1d).values.astype("datetime64[ns]")
            return base[:, None] + lead_vals.astype("timedelta64[h]")[None, :]

    if "forecast_reference_time" in ds_ifs.coords:
        frt_raw = ds_ifs["forecast_reference_time"].values
        frt_1d = np.asarray(frt_raw).reshape(-1)
        if frt_1d.shape[0] != n_rec:
            raise ValueError(
                f"forecast_reference_time len={frt_1d.shape[0]} != n_rec={n_rec}"
            )
        base = pd.to_datetime(frt_1d).values.astype("datetime64[ns]")
        return base[:, None] + lead_vals.astype("timedelta64[h]")[None, :]

    if "time" in ds_ifs.coords:
        tt_raw = ds_ifs["time"].values
        tt_arr = np.asarray(tt_raw)

        if tt_arr.ndim == 2:
            if tt_arr.shape == (n_rec, n_lead):
                return pd.to_datetime(tt_arr).values.astype("datetime64[ns]")
            if tt_arr.shape == (n_lead, n_rec):
                return pd.to_datetime(tt_arr.T).values.astype("datetime64[ns]")
            raise ValueError(f"2D time shape={tt_arr.shape}, expected {(n_rec, n_lead)}")

        if tt_arr.ndim == 1 and tt_arr.shape[0] == n_rec * n_lead:
            return pd.to_datetime(tt_arr).values.astype("datetime64[ns]").reshape(n_rec, n_lead)

        tt_1d = tt_arr.reshape(-1)
        if tt_1d.shape[0] == n_rec:
            base = pd.to_datetime(tt_1d).values.astype("datetime64[ns]")
            return base[:, None] + lead_vals.astype("timedelta64[h]")[None, :]

    raise ValueError(
        "Cannot recover IFS valid_time grid. Need valid_time(record,lead), "
        "valid_time(record), forecast_reference_time(record), or time."
    )


def load_ifs_long(
    ifs_nc: str, debug: bool = True
) -> pd.DataFrame:
    if not os.path.exists(ifs_nc):
        raise FileNotFoundError(ifs_nc)

    ds_ifs = xr.open_dataset(ifs_nc)
    try:
        if "VIS_ifs" not in ds_ifs:
            raise ValueError("IFS file must contain 'VIS_ifs'")

        vis_ifs = ds_ifs["VIS_ifs"].values
        if vis_ifs.ndim != 3:
            raise ValueError(
                f"VIS_ifs must be 3D (record, lead, station), got shape={vis_ifs.shape}"
            )

        n_rec, n_lead, n_st = vis_ifs.shape

        if "lead_hour" not in ds_ifs.coords:
            raise ValueError("IFS file must contain 'lead_hour' coord")
        lead_vals_raw = np.asarray(ds_ifs["lead_hour"].values)
        lead_vals = lead_vals_raw.reshape(-1).astype(np.int32)
        if lead_vals.shape[0] != n_lead:
            raise ValueError(
                f"lead_hour flattened len={lead_vals.shape[0]} != n_lead={n_lead}, "
                f"raw shape={lead_vals_raw.shape}"
            )

        if "station" not in ds_ifs.coords:
            raise ValueError("IFS file must contain 'station' coord")
        stations = normalize_station_id(ds_ifs["station"].values)
        if len(stations) != n_st:
            raise ValueError(f"station len={len(stations)} != n_st={n_st}")

        if debug:
            print("[IFS DEBUG] VIS_ifs shape:", vis_ifs.shape)
            print("[IFS DEBUG] lead_hour raw shape:", lead_vals_raw.shape)
            if "valid_time" in ds_ifs.coords:
                print("[IFS DEBUG] valid_time raw shape:", np.asarray(ds_ifs["valid_time"].values).shape)
            if "forecast_reference_time" in ds_ifs.coords:
                print(
                    "[IFS DEBUG] forecast_reference_time raw shape:",
                    np.asarray(ds_ifs["forecast_reference_time"].values).shape,
                )

        valid_time_grid = recover_valid_time_grid(ds_ifs, n_rec, n_lead, lead_vals)
        if valid_time_grid.shape != (n_rec, n_lead):
            raise ValueError(
                f"Recovered valid_time_grid shape={valid_time_grid.shape}, expected {(n_rec, n_lead)}"
            )

        valid_time_long = np.repeat(valid_time_grid.reshape(-1), n_st)
        lead_idx = np.tile(np.repeat(np.arange(n_lead), n_st), n_rec)
        station_idx = np.tile(np.arange(n_st), n_rec * n_lead)

        df = pd.DataFrame(
            {
                "time": pd.to_datetime(valid_time_long),
                "lead_hour": lead_vals[lead_idx].astype(np.int32),
                "station_id_norm": stations[station_idx].astype(str),
                "vis_ifs": vis_ifs.reshape(-1).astype(np.float64),
            }
        )

        df = df[np.isfinite(df["vis_ifs"])].copy()
        df["ifs_cls"] = df["vis_ifs"].map(vis_meters_to_obs_cls).astype(np.int32)
        df = df[df["ifs_cls"].isin([0, 1, 2])].copy()

        df["join_key"] = (
            df["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
            + "__"
            + df["station_id_norm"]
            + "__"
            + df["lead_hour"].astype(str)
        )

        dup_n = int(df["join_key"].duplicated().sum())
        if dup_n > 0:
            print(f"[IFS] duplicate exact keys before dedup: {dup_n}")
            df = df.drop_duplicates("join_key", keep="first").copy()

        return df.reset_index(drop=True)
    finally:
        ds_ifs.close()


def compute_metrics_from_hard_labels(
    y_true_cls: np.ndarray,
    y_pred_cls: np.ndarray,
    fog_th: float,
    mist_th: float,
    compute_rare_event_report: Callable,
) -> Dict[str, float]:
    dummy_probs = np.full((len(y_true_cls), 3), 1.0 / 3.0, dtype=np.float32)
    rep = compute_rare_event_report(
        dummy_probs,
        np.asarray(y_true_cls, dtype=np.int64),
        fog_th,
        mist_th,
        pred=np.asarray(y_pred_cls, dtype=np.int64),
    )
    keep_cols = [
        "Fog_CSI",
        "Fog_R",
        "Fog_P",
        "Mist_CSI",
        "Mist_R",
        "Mist_P",
        "low_vis_precision",
        "false_positive_rate",
    ]
    out = {}
    for c in keep_cols:
        out[c] = float(rep.get(c, np.nan))
    return out


def build_model_metrics_from_predictions(
    meta_sub: pd.DataFrame,
    y_cls: np.ndarray,
    pred_cls: np.ndarray,
    fog_th: float,
    mist_th: float,
    compute_rare_event_report: Callable,
) -> pd.DataFrame:
    df = meta_sub.copy()
    df["lead_hour"] = pd.to_numeric(df["lead_hour"], errors="coerce")
    df = df[df["lead_hour"].notna()].copy()
    df["lead_hour"] = np.rint(df["lead_hour"]).astype(np.int32)

    pred_cls = np.asarray(pred_cls).reshape(-1)
    y_cls = np.asarray(y_cls).reshape(-1)
    if len(pred_cls) != len(df) or len(y_cls) != len(df):
        raise ValueError("Length mismatch among meta_sub, y_cls, pred_cls")

    df["y_true"] = y_cls.astype(np.int32)
    df["y_pred"] = pred_cls.astype(np.int32)

    rows = []
    for h, sub in df.groupby("lead_hour"):
        row = {"lead_hour": int(h), "n_model": int(len(sub))}
        row.update(
            compute_metrics_from_hard_labels(
                sub["y_true"].values,
                sub["y_pred"].values,
                fog_th,
                mist_th,
                compute_rare_event_report,
            )
        )
        rows.append(row)

    return pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)


def run_model_vs_ifs_eval(
    meta_sub: pd.DataFrame,
    y_cls: np.ndarray,
    compute_rare_event_report: Callable,
    fog_th: float,
    mist_th: float,
    output_dir: str,
    ifs_all_leads_nc: str,
    lead_df: Optional[pd.DataFrame] = None,
    pred_cls: Optional[np.ndarray] = None,
    debug: bool = True,
) -> Dict[str, pd.DataFrame]:
    os.makedirs(output_dir, exist_ok=True)

    df_model_side = build_model_side(meta_sub, y_cls)
    print("[MODEL] rows:", len(df_model_side))
    print("[MODEL] unique join keys:", df_model_side["join_key"].nunique())
    print(
        "[MODEL] lead range:",
        df_model_side["lead_hour"].min(),
        "to",
        df_model_side["lead_hour"].max(),
    )

    df_ifs = load_ifs_long(ifs_all_leads_nc, debug=debug)
    print("[IFS] rows:", len(df_ifs))
    print("[IFS] unique join keys:", df_ifs["join_key"].nunique())
    print("[IFS] lead range:", df_ifs["lead_hour"].min(), "to", df_ifs["lead_hour"].max())

    df_join = df_model_side.merge(
        df_ifs[["join_key", "vis_ifs", "ifs_cls"]],
        on="join_key",
        how="left",
    )

    match_n = int(df_join["ifs_cls"].notna().sum())
    match_ratio = match_n / max(len(df_join), 1)
    print(f"[JOIN] exact matches: {match_n} / {len(df_join)} = {match_ratio:.2%}")

    if match_n == 0:
        raise RuntimeError(
            "No exact (time, station, lead) matches between model samples and IFS. "
            "Please check time zone, lead definition, and station normalization."
        )

    df_join_ok = df_join[df_join["ifs_cls"].notna()].copy()
    df_join_ok["ifs_cls"] = df_join_ok["ifs_cls"].astype(np.int32)

    diag_df = pd.DataFrame(
        [
            {
                "n_model_rows": int(len(df_model_side)),
                "n_model_unique_keys": int(df_model_side["join_key"].nunique()),
                "n_ifs_rows": int(len(df_ifs)),
                "n_ifs_unique_keys": int(df_ifs["join_key"].nunique()),
                "n_exact_matches": int(match_n),
                "exact_match_ratio": float(match_ratio),
            }
        ]
    )
    diag_path = os.path.join(output_dir, "lead_eval_alignment_diagnostics.csv")
    diag_df.to_csv(diag_path, index=False)
    print("[SAVE]", diag_path)

    rows = []
    for h, sub in df_join_ok.groupby("lead_hour"):
        if len(sub) < 30:
            continue
        row = {"lead_hour": int(h), "n_ifs": int(len(sub))}
        row.update(
            compute_metrics_from_hard_labels(
                sub["obs_cls"].values,
                sub["ifs_cls"].values,
                fog_th,
                mist_th,
                compute_rare_event_report,
            )
        )
        rows.append(row)

    ifs_lead_df = pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)
    ifs_csv = os.path.join(output_dir, "ifs_metrics_by_lead_hour.csv")
    ifs_lead_df.to_csv(ifs_csv, index=False)
    print("[SAVE]", ifs_csv)

    if lead_df is not None and isinstance(lead_df, pd.DataFrame) and len(lead_df) > 0:
        model_lead_df = lead_df.copy()
        if "n" in model_lead_df.columns and "n_model" not in model_lead_df.columns:
            model_lead_df = model_lead_df.rename(columns={"n": "n_model"})
    else:
        if pred_cls is None:
            raise RuntimeError(
                "lead_df is missing and pred_cls is missing. "
                "Please pass one of them into run_model_vs_ifs_eval()."
            )
        model_lead_df = build_model_metrics_from_predictions(
            meta_sub,
            y_cls,
            pred_cls,
            fog_th,
            mist_th,
            compute_rare_event_report,
        )

    model_csv = os.path.join(output_dir, "model_metrics_by_lead_hour.csv")
    model_lead_df.to_csv(model_csv, index=False)
    print("[SAVE]", model_csv)

    cmp_df = model_lead_df.merge(
        ifs_lead_df,
        on="lead_hour",
        how="inner",
        suffixes=("_model", "_ifs"),
    ).sort_values("lead_hour").reset_index(drop=True)

    if len(cmp_df) == 0:
        raise RuntimeError("Merged model-vs-IFS lead table is empty.")

    metric_cols = [
        "Fog_CSI",
        "Fog_R",
        "Fog_P",
        "Mist_CSI",
        "Mist_R",
        "Mist_P",
        "low_vis_precision",
        "false_positive_rate",
    ]
    for metric in metric_cols:
        m_col = f"{metric}_model"
        i_col = f"{metric}_ifs"
        if m_col in cmp_df.columns and i_col in cmp_df.columns:
            cmp_df[f"{metric}_diff_model_minus_ifs"] = cmp_df[m_col] - cmp_df[i_col]

    cmp_csv = os.path.join(output_dir, "model_vs_ifs_metrics_by_lead_hour.csv")
    cmp_df.to_csv(cmp_csv, index=False)
    print("[SAVE]", cmp_csv)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    axes[0, 0].plot(cmp_df["lead_hour"], cmp_df["Fog_CSI_model"], "o-", label="Model")
    axes[0, 0].plot(cmp_df["lead_hour"], cmp_df["Fog_CSI_ifs"], "o--", label="IFS")
    axes[0, 0].set_title("Fog CSI vs Lead")
    axes[0, 0].set_xlabel("Lead Hour")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(cmp_df["lead_hour"], cmp_df["Fog_R_model"], "o-", label="Model")
    axes[0, 1].plot(cmp_df["lead_hour"], cmp_df["Fog_R_ifs"], "o--", label="IFS")
    axes[0, 1].set_title("Fog Recall vs Lead")
    axes[0, 1].set_xlabel("Lead Hour")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(cmp_df["lead_hour"], cmp_df["Mist_P_model"], "o-", label="Model")
    axes[1, 0].plot(cmp_df["lead_hour"], cmp_df["Mist_P_ifs"], "o--", label="IFS")
    axes[1, 0].set_title("Mist Precision vs Lead")
    axes[1, 0].set_xlabel("Lead Hour")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(cmp_df["lead_hour"], cmp_df["Mist_R_model"], "o-", label="Model")
    axes[1, 1].plot(cmp_df["lead_hour"], cmp_df["Mist_R_ifs"], "o--", label="IFS")
    axes[1, 1].set_title("Mist Recall vs Lead")
    axes[1, 1].set_xlabel("Lead Hour")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    fig_path = os.path.join(output_dir, "fig_model_vs_ifs_by_lead.png")
    plt.savefig(fig_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("[SAVE]", fig_path)

    print("\nEvaluation complete.")
    print("Use 'model_vs_ifs_metrics_by_lead_hour.csv' as the main result table.")
    print("Do not use cross-lead aggregated precision as the main conclusion.")

    return {
        "df_model_side": df_model_side,
        "df_ifs": df_ifs,
        "df_join_ok": df_join_ok,
        "diag_df": diag_df,
        "ifs_lead_df": ifs_lead_df,
        "model_lead_df": model_lead_df,
        "cmp_df": cmp_df,
    }


# Notebook usage example:
#
# from eval_model_vs_ifs_by_lead import run_model_vs_ifs_eval
#
# results = run_model_vs_ifs_eval(
#     meta_sub=meta_sub,
#     y_cls=y_cls,
#     compute_rare_event_report=compute_rare_event_report,
#     fog_th=fog_th,
#     mist_th=mist_th,
#     output_dir=output_dir,
#     ifs_all_leads_nc=ifs_all_leads_nc,
#     lead_df=lead_df,      # or pred_cls=pred_cls
#     pred_cls=None,
#     debug=True,
# )
#
# display(results["cmp_df"])


results = run_model_vs_ifs_eval(
    meta_sub=meta_sub,
    y_cls=y_cls,
    compute_rare_event_report=compute_rare_event_report,
    fog_th=fog_th,
    mist_th=mist_th,
    output_dir=output_dir,
    ifs_all_leads_nc=ifs_all_leads_nc,
    lead_df=lead_df,
    pred_cls=None,
    debug=True,
)

cmp_df = results["cmp_df"]
ifs_lead_df = results["ifs_lead_df"]
display(cmp_df)


In [ ]:
# ========= Threshold sweep: CSI vs composite Target Achievement (TA) =========
# Training selects thresholds by maximizing TA (weighted recall/accuracy/precision/FPR),
# NOT CSI. This figure explains why operational thresholds may differ from CSI peaks.

TA_CONFIG = {
    'TARGET_RECALL_500_GOAL': 0.65,  'TARGET_W_RECALL_500': 0.30,
    'TARGET_RECALL_1000_GOAL': 0.75, 'TARGET_W_RECALL_1000': 0.30,
    'TARGET_ACCURACY_GOAL': 0.95,    'TARGET_W_ACCURACY': 0.25,
    'TARGET_LOW_VIS_PREC_GOAL': 0.20,'TARGET_W_LOW_VIS_PREC': 0.10,
    'TARGET_FPR_GOAL': 0.40,         'TARGET_W_FPR': 0.05,
}


def _sweep_one(probs_a, y_a, f_th, m_th):
    """Compute CSI + TA for one (f_th, m_th) pair using mutual rule."""
    n = len(y_a)
    ps = np.full(n, 2, dtype=np.int32)
    ps[(probs_a[:, 0] > f_th) & (probs_a[:, 0] > probs_a[:, 1])] = 0
    ps[(probs_a[:, 1] > m_th) & (probs_a[:, 1] > probs_a[:, 0])] = 1

    is_f, is_m, is_c = y_a == 0, y_a == 1, y_a == 2
    pf, pm = ps == 0, ps == 1

    tp_f = int((pf & is_f).sum())
    fog_csi = tp_f / max(tp_f + int((pf & ~is_f).sum()) + int((~pf & is_f).sum()), 1)
    fog_r = tp_f / max(tp_f + int((~pf & is_f).sum()), 1)
    fog_p = tp_f / max(tp_f + int((pf & ~is_f).sum()), 1)

    tp_m = int((pm & is_m).sum())
    mist_csi = tp_m / max(tp_m + int((pm & ~is_m).sum()) + int((~pm & is_m).sum()), 1)
    mist_r = tp_m / max(tp_m + int((~pm & is_m).sum()), 1)
    mist_p = tp_m / max(tp_m + int((pm & ~is_m).sum()), 1)

    clear_r = int(((ps == 2) & is_c).sum()) / max(int(is_c.sum()), 1)
    acc = float((ps == y_a).mean())
    pl, tl = ps <= 1, y_a <= 1
    lv_tp = int((pl & tl).sum())
    lv_p = lv_tp / max(lv_tp + int((pl & ~tl).sum()), 1)
    fpr_v = int((pl & is_c).sum()) / max(int(is_c.sum()), 1)

    c = TA_CONFIG
    ta = (min(fog_r / c['TARGET_RECALL_500_GOAL'], 1) * c['TARGET_W_RECALL_500']
          + min(mist_r / c['TARGET_RECALL_1000_GOAL'], 1) * c['TARGET_W_RECALL_1000']
          + min(acc / c['TARGET_ACCURACY_GOAL'], 1) * c['TARGET_W_ACCURACY']
          + min(lv_p / c['TARGET_LOW_VIS_PREC_GOAL'], 1) * c['TARGET_W_LOW_VIS_PREC']
          + min((1 - fpr_v) / (1 - c['TARGET_FPR_GOAL']), 1) * c['TARGET_W_FPR'])

    return dict(fog_csi=fog_csi, mist_csi=mist_csi, fog_p=fog_p, mist_p=mist_p,
                clear_r=clear_r, ta=ta,
                ok=fog_p >= 0.10 and mist_p >= 0.10 and clear_r >= 0.90)


th_g = np.linspace(0.05, 0.90, 51)

fog_res = [_sweep_one(probs, y_cls, ft, mist_th) for ft in th_g]
f_csi = np.array([r['fog_csi'] for r in fog_res])
f_ta = np.array([r['ta'] for r in fog_res])
f_ok = np.array([r['ok'] for r in fog_res], dtype=bool)

mist_res = [_sweep_one(probs, y_cls, fog_th, mt) for mt in th_g]
m_csi = np.array([r['mist_csi'] for r in mist_res])
m_ta = np.array([r['ta'] for r in mist_res])
m_ok = np.array([r['ok'] for r in mist_res], dtype=bool)

CF, CM, CT, CG, CP = "#2E5A87", "#D97706", "#4B5563", "#D1FAE5", "#059669"

setup_paper_style(font_family="DejaVu Sans")
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))


def _panel(ax, th, vals, ok, op_th, color, title, panel_lbl, is_ta=False):
    ym = max(float(vals.max()) * 1.12, 0.01)
    if ok.any():
        ax.fill_between(th, 0, ym, where=ok, color=CG, alpha=0.45,
                        label='Constraints met', zorder=1)
    ax.plot(th, vals, color=CT if is_ta else color, lw=2.2, zorder=3)
    ip = int(np.argmax(vals))
    mc = CP if is_ta else '#DC2626'
    pk_name = 'TA' if is_ta else 'CSI'
    ax.plot(th[ip], vals[ip], 'v', color=mc, ms=9, zorder=5,
            label=f'{pk_name} peak (th={th[ip]:.2f})')
    ax.axvline(op_th, color=color, ls='--', lw=1.5, alpha=0.7, zorder=4,
               label=f'Operational ({op_th:.2f})')
    ax.set_xlim(th[0], th[-1])
    ax.set_ylim(0, ym)
    ax.set_title(f'({panel_lbl}) {title}', fontsize=11, fontweight='600')
    ax.legend(fontsize=7.5, loc='best', framealpha=0.92)


_panel(axes[0, 0], th_g, f_csi, f_ok, fog_th, CF, 'Fog CSI vs threshold', 'a')
axes[0, 0].set_xlabel('Fog threshold')
axes[0, 0].set_ylabel('Fog CSI')

_panel(axes[0, 1], th_g, m_csi, m_ok, mist_th, CM, 'Mist CSI vs threshold', 'b')
axes[0, 1].set_xlabel('Mist threshold')
axes[0, 1].set_ylabel('Mist CSI')

_panel(axes[1, 0], th_g, f_ta, f_ok, fog_th, CF, 'Composite TA vs fog threshold', 'c', is_ta=True)
axes[1, 0].set_xlabel('Fog threshold')
axes[1, 0].set_ylabel('Target Achievement')

_panel(axes[1, 1], th_g, m_ta, m_ok, mist_th, CM, 'Composite TA vs mist threshold', 'd', is_ta=True)
axes[1, 1].set_xlabel('Mist threshold')
axes[1, 1].set_ylabel('Target Achievement')

fig.suptitle(
    'Threshold sweep: CSI (top) vs composite Target Achievement (bottom)\n'
    'Operational thresholds maximize TA under precision/recall constraints (green)',
    fontsize=11, fontweight='600', y=1.02)
plt.tight_layout()
_p = os.path.join(output_dir, "fig_threshold_sweep_csi_vs_ta.png")
save_figure(fig, _p)
plt.show()
print(f"[Sweep] Fog CSI peak@{th_g[np.argmax(f_csi)]:.2f}, "
      f"TA peak@{th_g[np.argmax(f_ta)]:.2f}, operational={fog_th}")
print(f"[Sweep] Mist CSI peak@{th_g[np.argmax(m_csi)]:.2f}, "
      f"TA peak@{th_g[np.argmax(m_ta)]:.2f}, operational={mist_th}")

In [ ]:
# ========= Scenario evaluation: 3 separate figures (time-of-day / season / region) =========

meta_scenario = derive_scenario_columns(meta_sub.copy())

METRIC_KEYS = ['fog_csi', 'fog_pod', 'mist_csi', 'lv_precision', 'macro_f1']
METRIC_LABELS = ['Fog CSI', 'Fog Recall', 'Mist CSI', 'Low-vis Prec', 'Macro-F1']
BAR_COLORS = ['#2E5A87', '#7BA7CC', '#D97706', '#6B7280', '#374151']


def _grouped_bars(ax, names, metrics_dict, title, xlabel_rot=0):
    """Draw grouped bar chart for multiple metrics across scenarios."""
    x = np.arange(len(names))
    n_m = len(METRIC_KEYS)
    w = 0.80 / n_m
    for i, (key, label, color) in enumerate(zip(METRIC_KEYS, METRIC_LABELS, BAR_COLORS)):
        vals = []
        for g in names:
            v = metrics_dict[g].get(key, np.nan) if g in metrics_dict else np.nan
            vals.append(float(v) if isinstance(v, (int, float)) and np.isfinite(v) else 0.0)
        offset = (i - (n_m - 1) / 2) * w
        ax.bar(x + offset, vals, w, color=color, label=label,
               edgecolor='white', lw=0.5)
    ax.set_xticks(x)
    ha = 'right' if xlabel_rot else 'center'
    ax.set_xticklabels(names, rotation=xlabel_rot, ha=ha, fontsize=9)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.0)
    ax.set_title(title, fontsize=12, fontweight='600', pad=8)
    ax.legend(fontsize=7.5, ncol=3, loc='upper right', framealpha=0.92)
    ax.grid(axis='y', alpha=0.25)


# ---- (1) Time-of-day ----
hour_arr = meta_scenario['hour'].values
tod_defs = [
    ('Morning (06-12h)', (hour_arr >= 6) & (hour_arr < 12)),
    ('Afternoon (12-18h)', (hour_arr >= 12) & (hour_arr < 18)),
    ('Evening (18-24h)', hour_arr >= 18),
    ('Night (00-06h)', (hour_arr >= 0) & (hour_arr < 6)),
]
tod_m = {}
for name, mask in tod_defs:
    if mask.sum() >= 50:
        tod_m[name] = _compute_scenario_metrics(y_cls[mask], preds[mask])

setup_paper_style(font_family="DejaVu Sans")
fig1, ax1 = plt.subplots(figsize=(8.5, 5))
_grouped_bars(ax1, [n for n, _ in tod_defs if n in tod_m], tod_m,
              'Model performance by time of day')
plt.tight_layout()
save_figure(fig1, os.path.join(output_dir, "fig_scenario_time_of_day.png"))
plt.show()

# ---- (2) Season ----
month_arr = meta_scenario['month'].values
season_defs = [
    ('DJF (Dec-Feb)', (month_arr == 12) | (month_arr <= 2)),
    ('MAM (Mar-May)', (month_arr >= 3) & (month_arr <= 5)),
    ('JJA (Jun-Aug)', (month_arr >= 6) & (month_arr <= 8)),
    ('SON (Sep-Nov)', (month_arr >= 9) & (month_arr <= 11)),
]
ssn_m = {}
for name, mask in season_defs:
    if mask.sum() >= 50:
        ssn_m[name] = _compute_scenario_metrics(y_cls[mask], preds[mask])

fig2, ax2 = plt.subplots(figsize=(8.5, 5))
_grouped_bars(ax2, [n for n, _ in season_defs if n in ssn_m], ssn_m,
              'Model performance by season')
plt.tight_layout()
save_figure(fig2, os.path.join(output_dir, "fig_scenario_season.png"))
plt.show()

# ---- (3) Region ----
region_arr = meta_scenario['region'].values
is_coast = meta_scenario['is_coastal'].values
region_names_sorted = sorted([r for r in np.unique(region_arr) if r != 'Other'])
reg_defs = [(rn, region_arr == rn) for rn in region_names_sorted]
reg_defs.append(('Coastal', is_coast == 1))
reg_defs.append(('Inland', is_coast == 0))

reg_m = {}
for name, mask in reg_defs:
    if mask.sum() >= 50:
        reg_m[name] = _compute_scenario_metrics(y_cls[mask], preds[mask])

fig3, ax3 = plt.subplots(figsize=(13, 5.5))
_grouped_bars(ax3, [n for n, _ in reg_defs if n in reg_m], reg_m,
              'Model performance by region', xlabel_rot=25)
plt.tight_layout()
save_figure(fig3, os.path.join(output_dir, "fig_scenario_region.png"))
plt.show()

# Save combined CSV
all_scn = {}
for prefix, d in [('ToD', tod_m), ('Season', ssn_m), ('Region', reg_m)]:
    for k, v in d.items():
        all_scn[f'{prefix}: {k}'] = v
rows_csv = []
for name, m in all_scn.items():
    row = {'scenario': name}
    row.update(m)
    rows_csv.append(row)
csv_p = os.path.join(output_dir, "scenario_metrics_three_views.csv")
pd.DataFrame(rows_csv).to_csv(csv_p, index=False, float_format="%.4f")
print(f"[Scenario] 3 figures + CSV saved -> {csv_p}")

In [ ]:
# ========= Three widespread fog events: footprint row + peak maps (same detector as paper_eval) =========

meta_evt = meta_sub.copy()
if "station_id" not in meta_evt.columns:
    meta_evt["station_id"] = meta_evt["station_id_norm"]

shp_gdf = load_china_shapefile(EVENT_SHP_PATH) if EVENT_SHP_PATH else None

event_df = pd.DataFrame()
for (nf, nr, lon_s, lat_s) in [
    (80, 3, 10.0, 4.0),
    (50, 2, 8.0, 3.0),
    (30, 2, 5.0, 2.0),
    (15, 1, 3.0, 1.5),
]:
    event_df = detect_widespread_fog_events(
        meta_evt,
        y_cls,
        top_k=3,
        min_fog_stations=nf,
        min_regions=nr,
        min_lon_span=lon_s,
        min_lat_span=lat_s,
        gap_hours=24,
    )
    if len(event_df) >= 3:
        break

if event_df.empty or len(event_df) < 3:
    print(f"[Events] detect_widespread_fog_events returned {len(event_df)} events (<3). Skip footprint/peak maps.")
else:
    event_df = event_df.sort_values("event_rank").reset_index(drop=True)
    event_df.to_csv(os.path.join(output_dir, "widespread_events_detected.csv"), index=False)
    print(event_df[["event_rank", "peak_time", "peak_fog_count", "peak_region_count"]])

    plot_three_events_footprint_row(
        meta_evt,
        y_raw_sub,
        preds,
        event_df,
        os.path.join(output_dir, "fig_three_events_footprint_row.png"),
        shp_gdf=shp_gdf,
        window_hours=EVENT_FOOTPRINT_WINDOW_H,
    )
    plot_three_events_peak_row(
        meta_evt,
        y_raw_sub,
        preds,
        event_df,
        os.path.join(output_dir, "fig_three_events_peak_row.png"),
        shp_gdf=shp_gdf,
    )
    print("[Events] Saved fig_three_events_footprint_row.png and fig_three_events_peak_row.png")